# Analysis — Charts and Tables

**Read-only.** This notebook never launches simulations. It loads stored scenario CSVs and per-seed Monte Carlo files, validates them, and writes tables/figures under `analysis/`.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
for p in [ROOT, ROOT / 'common', ROOT / 'cpp_engine']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from analysis import discover_runs, select_run, run_analysis

RUN_ID = None          # set explicitly to choose a run
INCLUDE_SMOKE = False  # keep smoke tests separate from real results


## Available runs

In [ ]:
runs = discover_runs(ROOT / 'results', include_smoke=INCLUDE_SMOKE)
print(json.dumps(runs, indent=2)[:6000])
if not runs:
    print('No runs found yet. Run a B1-B4 notebook or the CLI first.')


## Select and validate one run

In [ ]:
run_dir = None
try:
    run_dir = select_run(RUN_ID, results_root=ROOT / 'results', include_smoke=INCLUDE_SMOKE)
    print('Selected run:', run_dir)
except FileNotFoundError as exc:
    print('No run available:', exc)


## Build tables and figures

In [ ]:
result = None
if run_dir is not None:
    result = run_analysis(run_dir=run_dir, make_figures=True, n_boot=2000)
    print('Validation:', result['report']['ok'])
    print('Issues:', result['report']['issues'])
    print('Report:', result['report_path'])
    print('Figures:', result['figure_dir'])


## Tables

In [ ]:
from IPython.display import display
if result is not None:
    for name in ['overall', 'objective_ranking', 'mode_comparison', 'best_config', 'runtime', 'robustness']:
        print('##', name)
        display(result['tables'][name])


## Figures

In [ ]:
from IPython.display import Image, display
if result is not None:
    for path in result['figure_paths']:
        print(path)
        display(Image(filename=path))


## Interpretation guardrails

- Rank objectives by **final Net Advantage**, never by optimizer fitness.
- O2 (Competition-aware) is the only fully validated objective.
- O1/O3/O4/O5/O6 use the provisional paired B-only `Blocked_B` protocol; their results are exploratory.
- Monte Carlo replicates are common random numbers, not independent optimizer runs. Do not treat MC replicates as between-run replication.
- Swap vs SSA comparisons are paired by MC seed within the same scenario/K/objective.